## Task 1 — Dataset Loading and Data Understanding

### Prediction objective

Supervised **regression** to predict `TARGET_deathRate`: mean per-capita (per 100,000) cancer mortalities by county (2010–2016).

### Key columns (from data dictionary)

| Column | Meaning |
|---|---|
| `TARGET_deathRate` | Dependent variable; mean per-capita cancer mortalities |
| `avgAnnCount` | Mean annual reported cancer diagnoses |
| `avgDeathsPerYear` | Mean annual cancer mortalities |
| `incidenceRate` | Mean per-capita cancer diagnoses |
| `medIncome` | Median income per county |
| `popEst2015` | County population (2015) |
| `povertyPercent` | Percent of populace in poverty |
| `AvgHouseholdSize` | Mean household size |
| `Geography` | County name (identifier; not a model feature) |
| `binnedInc` | Median income binned by decile |
| Education / coverage / race % columns | ACS-style demographic and insurance shares |

In [2]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Project root is one level above notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import config as cfg

In [3]:
# Load cancer registry dataset
df = pd.read_csv(cfg.RAW_CSV, encoding=cfg.CSV_ENCODING)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

Shape: 3,047 rows × 34 columns


,avgAnnCount,avgDeathsPerYear,TARGET_deathRate,incidenceRate,medIncome,popEst2015,povertyPercent,studyPerCap,binnedInc,MedianAge,...,PctPrivateCoverageAlone,PctEmpPrivCoverage,PctPublicCoverage,PctPublicCoverageAlone,PctWhite,PctBlack,PctAsian,PctOtherRace,PctMarriedHouseholds,BirthRate
0,1397.0,469,164.9,489.8,61898,260131,11.2,499.748204,"(61494.5, 125635]",39.3,...,NaN,41.6,32.9,14.0,81.780529,2.594728,4.821857,1.843479,52.856076,6.118831
1,173.0,70,161.3,411.6,48127,43269,18.6,23.111234,"(48021.6, 51046.4]",33.0,...,53.8,43.6,31.1,15.3,89.228509,0.969102,2.246233,3.741352,45.372500,4.333096
2,102.0,50,174.7,349.7,49348,21026,14.6,47.560164,"(48021.6, 51046.4]",45.0,...,43.5,34.9,42.1,21.1,90.922190,0.739673,0.465898,2.747358,54.444868,3.729488
3,427.0,202,194.8,430.4,44243,75882,17.1,342.637253,"(42724.4, 45201]",42.8,...,40.3,35.0,45.3,25.0,91.744686,0.782626,1.161359,1.362643,51.021514,4.603841
4,57.0,26,144.4,350.1,49955,10321,12.5,0.000000,"(48021.6, 51046.4]",48.3,...,43.9,35.1,44.0,22.7,94.104024,0.270192,0.665830,0.492135,54.027460,6.796657


### Data validation

In [4]:
df.dtypes.to_frame("dtype")

,dtype
avgAnnCount,float64
avgDeathsPerYear,int64
TARGET_deathRate,float64
incidenceRate,float64
medIncome,int64
popEst2015,int64
povertyPercent,float64
studyPerCap,float64
binnedInc,object
MedianAge,float64


In [7]:
print("Missing values (nonzero only)")
missing = df.isna().sum()
missing_pct = (100 * missing / len(df)).round(2)
missing_tbl = (
    pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)
display(missing_tbl if len(missing_tbl) else pd.DataFrame({"note": ["No missing values"]}))

Missing values (nonzero only)


,missing_count,missing_pct
PctSomeCol18_24,2285,74.99
PctPrivateCoverageAlone,609,19.99
PctEmployed16_Over,152,4.99


In [8]:
n_dupes = int(df.duplicated().sum())
print(f"\nDuplicate rows: {n_dupes}")


Duplicate rows: 0


In [9]:
# Geography uniqueness (county-level ID)
if "Geography" in df.columns:
    print(f"Unique Geography values: {df['Geography'].nunique():,} / {len(df):,} rows")

Unique Geography values: 3,047 / 3,047 rows


In [10]:
# Target and key numeric summaries
key_cols = [
    cfg.TARGET,
    cfg.COL_MEDIAN_INCOME,
    cfg.COL_POVERTY,
    cfg.COL_AVG_HH_SIZE,
    cfg.MEDIAN_AGE_COL,
]
display(df[key_cols].describe().T)

# Invalid MedianAge (ages above a plausible max)
invalid_age_mask = df[cfg.MEDIAN_AGE_COL] > cfg.MEDIAN_AGE_MAX_VALID
n_invalid_age = int(invalid_age_mask.sum())
print(
    f"\nInvalid {cfg.MEDIAN_AGE_COL} > {cfg.MEDIAN_AGE_MAX_VALID}: "
    f"{n_invalid_age} rows (max={df[cfg.MEDIAN_AGE_COL].max():.1f})"
)
if n_invalid_age:
    display(df.loc[invalid_age_mask, ["Geography", cfg.MEDIAN_AGE_COL]].head(10))

,count,mean,std,min,25%,50%,75%,max
TARGET_deathRate,3047.0,178.664063,27.751511,59.7000,161.20,178.1,195.20,362.80
medIncome,3047.0,47063.281917,12040.090836,22640.0000,38882.50,45207.0,52492.00,125635.00
povertyPercent,3047.0,16.878175,6.409087,3.2000,12.15,15.9,20.40,47.40
AvgHouseholdSize,3047.0,2.479662,0.429174,0.0221,2.37,2.5,2.63,3.97
MedianAge,3047.0,45.272333,45.304480,22.3000,37.70,41.0,44.00,624.00



Invalid MedianAge > 100: 30 rows (max=624.0)


,Geography,MedianAge
100,"Seward County, Nebraska",458.4
181,"Sandoval County, New Mexico",469.2
225,"Pittsylvania County, Virginia",546.0
318,"Iosco County, Michigan",624.0
425,"Person County, North Carolina",508.8
606,"Mineral County, Montana",619.2
637,"Cass County, Nebraska",498.0
843,"Tangipahoa Parish, Louisiana",412.8
991,"Greene County, Virginia",481.2
1199,"Harrison County, Mississippi",424.8


In [5]:
# Non-numeric / categorical columns
non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
print("Non-numeric columns:", non_numeric)
for col in non_numeric:
    print(f"  {col}: nunique={df[col].nunique()}, sample={df[col].dropna().iloc[0]!r}")

Non-numeric columns: ['binnedInc', 'Geography']
  binnedInc: nunique=10, sample='(61494.5, 125635]'
  Geography: nunique=3047, sample='Kitsap County, Washington'


### Assumptions and columns excluded from modeling

**Exclusions (locked in `config.EXCLUDE_COLUMNS`):**

- `Geography` — high-cardinality county identifier; not a generalizable feature
- `binnedInc` — redundant with continuous `medIncome`
- `PctSomeCol18_24` — ~75% missing; dropping avoids heavy imputation bias

**Other assumptions for later preprocessing (Task 2):**

- Treat `MedianAge > 100` as missing, then median-impute (fit on train only)
- Median-impute remaining missing numeric columns (`PctEmployed16_Over`, `PctPrivateCoverageAlone`) on train only
- Keep `avgDeathsPerYear`, `avgAnnCount`, and `incidenceRate` as features for predictive performance, but note that they are strongly related to mortality and may overstate deployable accuracy